In [4]:
import os
import glob
import torch
from PIL import Image, ImageDraw, ImageFont
import numpy as np

class ColorMapper:
    def __init__(self):
        self.colors = {}

    def get_color(self, class_name):
        if class_name not in self.colors:
            h = hash(class_name)
            r = (h & 0xFF0000) >> 16
            g = (h & 0x00FF00) >> 8
            b = (h & 0x0000FF)
            self.colors[class_name] = (r, g, b)
        return self.colors[class_name]

def rotate_point_cw90(x, y, img_width, img_height):
    return img_height - 1 - y, x

def rotate_bbox_cw90(x1, y1, x2, y2, img_width, img_height):
    """
    Преобразует bbox (x1<=x2, y1<=y2) в повёрнутое изображение,
    используя все четыре угла.
    """
    corners = [(x1, y1), (x2, y1), (x1, y2), (x2, y2)]
    rotated = [rotate_point_cw90(cx, cy, img_width, img_height) for cx, cy in corners]
    xs = [p[0] for p in rotated]
    ys = [p[1] for p in rotated]
    return min(xs), min(ys), max(xs), max(ys)

def draw_annotations_on_image(image_path, pt_path, output_path=None, show=False, font_size=12, color_mapper=None):
    if color_mapper is None:
        color_mapper = ColorMapper()

    # Исходное изображение (до поворота)
    img_orig = Image.open(image_path).convert('RGB')
    W, H = img_orig.size   # W = ширина, H = высота

    # Загружаем PT-файл (формат: (cx, cy, w, h) относительные)
    data = torch.load(pt_path, map_location='cpu')
    nodes = data.get('x', [])   # список узлов

    # Поворачиваем изображение
    img_rot = img_orig.transpose(Image.ROTATE_270)  # 90° по часовой
    draw = ImageDraw.Draw(img_rot)

    try:
        font = ImageFont.truetype("arial.ttf", font_size)
    except:
        font = ImageFont.load_default()

    for node in nodes:
        cx_rel, cy_rel, w_rel, h_rel = node

        # Абсолютные координаты в исходном изображении
        x1 = (cx_rel - w_rel/2) * W
        y1 = (cy_rel - h_rel/2) * H
        x2 = (cx_rel + w_rel/2) * W
        y2 = (cy_rel + h_rel/2) * H

        # Преобразуем bbox в повёрнутую систему координат
        x1_rot, y1_rot, x2_rot, y2_rot = rotate_bbox_cw90(x1, y1, x2, y2, W, H)

        class_name = "test"   # укажите реальное имя, если есть
        color = color_mapper.get_color(class_name)

        # Рисуем прямоугольник
        draw.rectangle([x1_rot, y1_rot, x2_rot, y2_rot], outline=color, width=2)
        draw.text((x1_rot + 5, y1_rot + 5), class_name, fill=color, font=font)

        # Центр объекта
        cx_abs = cx_rel * W
        cy_abs = cy_rel * H
        cx_rot, cy_rot = rotate_point_cw90(cx_abs, cy_abs, W, H)
        r = 4
        draw.ellipse([cx_rot - r, cy_rot - r, cx_rot + r, cy_rot + r], fill='red', outline='red')

    if output_path:
        img_rot.save(output_path)
        print(f"Saved: {output_path}")
    if show:
        import matplotlib.pyplot as plt
        plt.imshow(np.array(img_rot))
        plt.axis('off')
        plt.title(os.path.basename(image_path))
        plt.show()

def visualize_all_annotations(image_root, pt_root, output_root=None, show=False, verbose=True, max_scenes=None):
    """
    max_scenes: если задано, обрабатывает только первые max_scenes сцен.
    """
    image_scenes_dir = os.path.join(image_root, 'scenes')
    if not os.path.isdir(image_scenes_dir):
        print(f"Папка scenes не найдена: {image_scenes_dir}")
        return

    scene_names = [d for d in os.listdir(image_scenes_dir) if os.path.isdir(os.path.join(image_scenes_dir, d))]
    if max_scenes is not None:
        scene_names = scene_names[:max_scenes]
    if verbose:
        print(f"Найдено сцен: {len(scene_names)}")

    color_mapper = ColorMapper()

    for scene in scene_names:
        if scene != "0958220d-e2c2-2de1-9710-c37018da1883":
            continue
        scene_image_path = os.path.join(image_scenes_dir, scene, 'sequence')
        if not os.path.isdir(scene_image_path):
            continue

        scene_pt_path = os.path.join(pt_root, scene)
        if not os.path.isdir(scene_pt_path):
            if verbose:
                print(f"PT папка для сцены {scene} не найдена: {scene_pt_path}")
            continue

        image_pattern = os.path.join(scene_image_path, "**", "frame-*.color.jpg")
        image_paths = glob.glob(image_pattern, recursive=True)
        image_paths = [p for p in image_paths if '.rendered.' not in p]

        if verbose:
            print(f"Сцена {scene}: {len(image_paths)} изображений")

        for img_path in image_paths:
            base_name = os.path.basename(img_path)
            pt_name = base_name.replace('.color.jpg', '.pt')
            # Ищем .pt файл в scene_pt_path рекурсивно
            pt_pattern = os.path.join(scene_pt_path, "**", pt_name)
            candidates = glob.glob(pt_pattern, recursive=True)
            if not candidates:
                if verbose:
                    print(f"pt не найден: {pt_name} в {scene_pt_path}")
                continue
            pt_path = candidates[0]

            if output_root:
                rel_path = os.path.relpath(img_path, image_root)
                output_path = os.path.join(output_root, rel_path)
                os.makedirs(os.path.dirname(output_path), exist_ok=True)
                output_path = output_path.replace('.color.jpg', '.annotated.jpg')
            else:
                output_path = None

            draw_annotations_on_image(img_path, pt_path, output_path, show, color_mapper=color_mapper)

image_root = "/mnt/external_usb_hdd/6YL/Datasets/3RScan"
pt_root   = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/Splited_graphs"
output_root = "/home/pinkin_ek/projects/Scene_graph_localization/data/graph_vis_rotated"
# Для теста обработаем только первые 2 сцены
visualize_all_annotations(image_root, pt_root, output_root, show=False, verbose=True, max_scenes=10)

Найдено сцен: 10
Сцена c92fb570-f771-2064-8700-e44ec1355c49: 107 изображений
Saved: /home/pinkin_ek/projects/Scene_graph_localization/data/graph_vis_rotated/scenes/c92fb570-f771-2064-8700-e44ec1355c49/sequence/frame-000040.annotated.jpg
Saved: /home/pinkin_ek/projects/Scene_graph_localization/data/graph_vis_rotated/scenes/c92fb570-f771-2064-8700-e44ec1355c49/sequence/frame-000077.annotated.jpg
Saved: /home/pinkin_ek/projects/Scene_graph_localization/data/graph_vis_rotated/scenes/c92fb570-f771-2064-8700-e44ec1355c49/sequence/frame-000030.annotated.jpg
Saved: /home/pinkin_ek/projects/Scene_graph_localization/data/graph_vis_rotated/scenes/c92fb570-f771-2064-8700-e44ec1355c49/sequence/frame-000001.annotated.jpg
Saved: /home/pinkin_ek/projects/Scene_graph_localization/data/graph_vis_rotated/scenes/c92fb570-f771-2064-8700-e44ec1355c49/sequence/frame-000087.annotated.jpg
Saved: /home/pinkin_ek/projects/Scene_graph_localization/data/graph_vis_rotated/scenes/c92fb570-f771-2064-8700-e44ec1355c49

/tmp/ipykernel_28670/1966790312.py:43: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(pt_path, map_location='cpu')


Saved: /home/pinkin_ek/projects/Scene_graph_localization/data/graph_vis_rotated/scenes/c92fb570-f771-2064-8700-e44ec1355c49/sequence/frame-000028.annotated.jpg
Saved: /home/pinkin_ek/projects/Scene_graph_localization/data/graph_vis_rotated/scenes/c92fb570-f771-2064-8700-e44ec1355c49/sequence/frame-000106.annotated.jpg
Saved: /home/pinkin_ek/projects/Scene_graph_localization/data/graph_vis_rotated/scenes/c92fb570-f771-2064-8700-e44ec1355c49/sequence/frame-000071.annotated.jpg
Saved: /home/pinkin_ek/projects/Scene_graph_localization/data/graph_vis_rotated/scenes/c92fb570-f771-2064-8700-e44ec1355c49/sequence/frame-000072.annotated.jpg
Saved: /home/pinkin_ek/projects/Scene_graph_localization/data/graph_vis_rotated/scenes/c92fb570-f771-2064-8700-e44ec1355c49/sequence/frame-000056.annotated.jpg
Saved: /home/pinkin_ek/projects/Scene_graph_localization/data/graph_vis_rotated/scenes/c92fb570-f771-2064-8700-e44ec1355c49/sequence/frame-000065.annotated.jpg
Saved: /home/pinkin_ek/projects/Scene_gr

In [ ]:
pt_graph = torch.load("/mnt/external_usb_hdd/6YL/Datasets/3RScan/Splited_graphs/4a9a43e2-7736-2874-841a-f239715be4c6/frame-000000.pt")
print(pt_graph)

In [ ]:
json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/SceneGraphs/4a9a43e2-7736-2874-841a-f239715be4c6/frame-000000.json"
with open(json_path, 'r') as f:
    data = json.load(f)
    json_graph = json.load("/mnt/external_usb_hdd/6YL/Datasets/3RScan/Splited_graphs/4a9a43e2-7736-2874-841a-f239715be4c6/frame-000000.pt")
    print(json_graph)